# NB06 -- `shap_hierarchy` reconstruction (SPEC-NB06-001)

Reconstructs the SHAP attribution hierarchy from the 73 pinned fixed
models, validates it column-by-column against the live `shap_hierarchy` (the per-column
oracle), and -- if bit-identical -- stands as the reproducible, notebook-owned generator
of the V6 baseline `8274ad5ba3c38e73`.

**Recipe (CHG-0011 §1-§4):** TreeSHAP via `booster.predict(DMatrix, pred_contribs=True)`.
Per (config, stage_code, feature_name): `shap_mean_abs` = mean over folds of |SHAP|
(n=1/fold); `shap_std` = across-fold SD of |SHAP|; `cv_shap` = shap_std/shap_mean_abs
(NULL when shap_mean_abs==0); `importance_rank` per (config,stage) by shap_mean_abs DESC,
feature_name ASC; `feature_type` from `feature_ml_final.feature_kind` (C1->C62ALL,
C8->C45CRUDE); `is_robust` = 0. **Aggregation idiom default = pandas** (ddof=1, float64);
numpy idiom (ddof=0, float32) is the L3-swap candidate.

**Non-destructive:** computes into `shap_hierarchy_staging`; live untouched unless the
ratified Branch-2 (V7) path fires.


In [1]:
import sqlite3, pickle, sys, hashlib
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from collections import defaultdict

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
from utils import DB_PATH, canonical_table_fingerprint, load_ml_dataset

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

PINS = {
    'C1': {'run_id': '4aee320b2817', 'n_folds': 44, 'n_features': 142, 'feature_set': 'C62ALL'},
    'C8': {'run_id': '6b493bc7ab33', 'n_folds': 29, 'n_features': 127, 'feature_set': 'C45CRUDE'},
}
STAGES = ('W0', 'W1', 'W2', 'W3')
BASELINE_FP = '8274ad5ba3c38e73'
print(f'DB: {DB_PATH}')


DB: C:\Users\leogr\Documents\Data Science\TCC\data\processed\weathering.db


## §1 Pre-flight (read-only; HALT-on-fail)

Checks the inputs before any compute. X-load itself deferred to §3 where `load_ml_dataset` is confirmed.


In [2]:
# --- §1.A Pin canonical models + sha256 verify + held_out_oil capture ---
def _sha256(p, chunk=65536):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(chunk), b''):
            h.update(c)
    return h.hexdigest()

pin_rows = cur.execute('''
    SELECT config_name, fold_idx, held_out_oil, artifact_path, sha256, n_features
    FROM looo_model_artifacts
    WHERE config_name IN ('C1','C8') AND run_id IN ('4aee320b2817','6b493bc7ab33')
    ORDER BY config_name, fold_idx
''').fetchall()
assert len(pin_rows) == 73, f'§1.A FAIL: {len(pin_rows)} pins, expected 73'

pinned = defaultdict(list)
sha_verified_count = 0
for cfg, fidx, oil, rel_path, stored_sha, n_feat_db in pin_rows:
    p = Path(rel_path) if Path(rel_path).is_absolute() else (REPO_ROOT / rel_path)
    assert p.exists(), f'§1.A FAIL: missing artifact {p}'
    if stored_sha is not None:
        actual = _sha256(p)
        assert actual == stored_sha, \
            f'§1.A FAIL: sha256 mismatch on {p}\n  stored: {stored_sha}\n  actual: {actual}'
        sha_verified_count += 1
    assert n_feat_db == PINS[cfg]['n_features'], \
        f'§1.A FAIL: {cfg} fold {fidx} n_features={n_feat_db}, expected {PINS[cfg]["n_features"]}'
    pinned[cfg].append((fidx, oil, p))
assert len(pinned['C1']) == 44 and len(pinned['C8']) == 29, '§1.A FAIL: per-config fold counts'
print(f'§1.A PASS: 73 models pinned, sha256-verified {sha_verified_count}/73, held_out_oil captured')

# --- §1.B Load models; verify feature_names identity across folds + counts ---
models, feature_order = defaultdict(dict), {}
for cfg, fold_list in pinned.items():
    ref = None
    for fidx, oil, p in fold_list:
        with open(p, 'rb') as f:
            m = pickle.load(f)
        booster = m.get_booster() if hasattr(m, 'get_booster') else m
        fn = booster.feature_names
        assert fn is not None, f'§1.B FAIL: {cfg} fold {fidx} booster.feature_names is None'
        if ref is None:
            ref = fn
        else:
            assert fn == ref, f'§1.B FAIL: {cfg} fold {fidx} feature_names diverge'
        models[cfg][fidx] = (booster, oil)
    assert len(ref) == PINS[cfg]['n_features'], \
        f'§1.B FAIL: {cfg} trained on {len(ref)} features, expected {PINS[cfg]["n_features"]}'
    feature_order[cfg] = ref
print('§1.B PASS: feature_names identical across folds; 142/127 confirmed; oil bound to booster')

# --- §1.C Insert-order + hierarchy_id map (staging copies ids explicitly) ---
id_map = {}
for cfg in PINS:
    rows = cur.execute('SELECT hierarchy_id, stage_code, feature_name FROM shap_hierarchy '
                       'WHERE config=? ORDER BY hierarchy_id', (cfg,)).fetchall()
    assert len(rows) == PINS[cfg]['n_features'] * 4, \
        f'§1.C FAIL: {cfg} {len(rows)} rows, expected {PINS[cfg]["n_features"]*4}'
    id_map[cfg] = rows
print('§1.C PASS: hierarchy_id <-> (stage, feature) map captured')

# --- §1.E Cross-table paranoia: held_out_oil set consistency ---
for cfg in PINS:
    pred_oils = set(r[0] for r in cur.execute(
        'SELECT DISTINCT oil_id FROM looo_predictions WHERE config_name=? AND run_id=?',
        (cfg, PINS[cfg]['run_id'])
    ))
    art_oils = set(oil for fidx, oil, p in pinned[cfg])
    assert pred_oils == art_oils, \
        f'§1.E FAIL: {cfg} oil set mismatch; only-artifacts={art_oils-pred_oils}; only-predictions={pred_oils-art_oils}'
print('§1.E PASS: held_out_oil set consistent across looo_model_artifacts x looo_predictions')

# --- §1.F Oracle intact (covers Invariante #9 recipe-drift via same check) ---
live_fp = canonical_table_fingerprint(con, 'shap_hierarchy')
assert live_fp == BASELINE_FP, \
    f'§1.F FAIL: live fingerprint {live_fp} != {BASELINE_FP} -- table or recipe changed since AI'
print(f'§1.F PASS: live oracle intact ({BASELINE_FP})')
print()
print('§1 PRE-FLIGHT PASS -- ready for §2 (staging) + §3 (compute)')


§1.A PASS: 73 models pinned, sha256-verified 73/73, held_out_oil captured
§1.B PASS: feature_names identical across folds; 142/127 confirmed; oil bound to booster
§1.C PASS: hierarchy_id <-> (stage, feature) map captured
§1.E PASS: held_out_oil set consistent across looo_model_artifacts x looo_predictions
§1.F PASS: live oracle intact (8274ad5ba3c38e73)

§1 PRE-FLIGHT PASS -- ready for §2 (staging) + §3 (compute)


## §2 Staging schema · §3 Compute + populate (pandas idiom default)


In [3]:
cur.execute('DROP TABLE IF EXISTS shap_hierarchy_staging')
cur.execute('''
    CREATE TABLE shap_hierarchy_staging (
        hierarchy_id    INTEGER PRIMARY KEY,
        config          TEXT NOT NULL DEFAULT 'C1',
        stage_code      TEXT NOT NULL,
        feature_name    TEXT NOT NULL,
        feature_type    TEXT NOT NULL,
        shap_mean_abs   REAL,
        shap_std        REAL,
        cv_shap         REAL,
        importance_rank INTEGER,
        is_robust       INTEGER NOT NULL DEFAULT 0
    )
''')
con.commit()
print('§2 PASS: shap_hierarchy_staging created (live untouched)')


§2 PASS: shap_hierarchy_staging created (live untouched)


In [4]:
# === §3a Per-fold |SHAP| via TreeSHAP on pinned models ===
records = []
for cfg in PINS:
    X, y, meta = load_ml_dataset(con, only_crude=(cfg == 'C8'))
    feats = feature_order[cfg]
    missing = set(feats) - set(X.columns)
    assert not missing, f'§3a FAIL: {cfg} X missing {len(missing)} features e.g. {list(missing)[:3]}'
    X = X[feats]
    for fidx, (booster, oil) in models[cfg].items():
        m = (meta['oil_id'] == oil).values
        X_oil = X[m]
        stages_oil = meta.loc[m, 'stage_code'].values
        assert len(X_oil) == 4 and set(stages_oil) == set(STAGES), \
            f'§3a FAIL: {cfg} oil {oil} stages={sorted(stages_oil)}, expected {STAGES}'
        dm = xgb.DMatrix(X_oil, feature_names=feats)
        abs_contribs = np.abs(booster.predict(dm, pred_contribs=True)[:, :-1])
        for i, stage in enumerate(stages_oil):
            for j, feat in enumerate(feats):
                records.append((cfg, stage, feat, fidx, float(abs_contribs[i, j])))
shap_long = pd.DataFrame(records, columns=['config','stage_code','feature_name','fold_idx','shap_abs'])
exp_records = 44*4*142 + 29*4*127
assert len(shap_long) == exp_records, f'§3a FAIL: {len(shap_long)} records, expected {exp_records}'
print(f'§3a PASS: {len(shap_long)} per-fold |SHAP| records (float64 widened from float32 pred_contribs)')


§3a PASS: 39724 per-fold |SHAP| records (float64 widened from float32 pred_contribs)


In [5]:
# === §3 helper + ftype bridge (DRY for default + §5 exhaustion loop) ===
ftype = {}
for cfg in PINS:
    for fname, kind in cur.execute(
        'SELECT feature_name, feature_kind FROM feature_ml_final WHERE config=?',
        (PINS[cfg]['feature_set'],)):
        ftype[(cfg, fname)] = kind
kinds_seen = set(ftype.values())
assert kinds_seen == {'compound', 'ratio'}, \
    f'§3 bridge anomaly: feature_kind values = {kinds_seen} (expected compound + ratio)'
print(f'ftype bridge built: {sum(1 for k in ftype if k[0]=="C1")} C1 + {sum(1 for k in ftype if k[0]=="C8")} C8 lookups; kinds={kinds_seen}')

def materialize_staging(sl):
    '''Aggregate shap_long -> staging recipe-faithful. Parameterized by row order
    (the ULP/fold lever for §5 exhaustion loop). Returns fingerprint.'''
    agg = (sl.groupby(['config','stage_code','feature_name'], sort=False)['shap_abs']
             .agg(shap_mean_abs='mean', shap_std='std').reset_index())
    assert agg['shap_mean_abs'].dtype == np.float64 and agg['shap_std'].dtype == np.float64, \
        f'idiom drift: dtypes = {agg["shap_mean_abs"].dtype}/{agg["shap_std"].dtype} (expected float64)'
    agg['cv_shap'] = agg['shap_std'] / agg['shap_mean_abs'].where(agg['shap_mean_abs'] != 0)
    agg = agg.sort_values(['config','stage_code','shap_mean_abs','feature_name'],
                          ascending=[True, True, False, True]).reset_index(drop=True)
    agg['importance_rank'] = agg.groupby(['config','stage_code'], sort=False).cumcount() + 1
    idx = agg.set_index(['config','stage_code','feature_name'])
    rows = []
    for cfg in PINS:
        for hid, st, ft in id_map[cfg]:
            r = idx.loc[(cfg, st, ft)]
            cv = None if pd.isna(r['cv_shap']) else float(r['cv_shap'])
            rows.append((hid, cfg, st, ft, ftype[(cfg, ft)],
                         float(r['shap_mean_abs']), float(r['shap_std']),
                         cv, int(r['importance_rank']), 0))
    cur.execute('DELETE FROM shap_hierarchy_staging')
    cur.executemany('INSERT INTO shap_hierarchy_staging VALUES (?,?,?,?,?,?,?,?,?,?)', rows)
    con.commit()
    return canonical_table_fingerprint(con, 'shap_hierarchy_staging')

print('Helper materialize_staging(sl) defined')


ftype bridge built: 142 C1 + 127 C8 lookups; kinds={'compound', 'ratio'}
Helper materialize_staging(sl) defined


In [6]:
# === §3c Default call: pandas idiom + fold_idx-ascending order ===
fp_default = materialize_staging(shap_long)
n_staging = cur.execute('SELECT COUNT(*) FROM shap_hierarchy_staging').fetchone()[0]
assert n_staging == 1076, f'§3c FAIL: staging has {n_staging} rows, expected 1076'
print(f'§3c PASS: staging populated (1076 rows, pandas idiom, fold_idx default order)')
print(f'  fp at default fold_idx order: {fp_default}')


§3c PASS: staging populated (1076 rows, pandas idiom, fold_idx default order)
  fp at default fold_idx order: d464fe0e1e061de0


## §4 Oracle validation -- L1 model-load · L2 additivity · L3 per-key typed diff · L4 fingerprint


In [7]:
# === §4-L1 Inputs sane ===
assert all(len(models[c]) == PINS[c]['n_folds'] for c in PINS), '§4-L1 FAIL: model count'
assert cur.execute('SELECT COUNT(*) FROM shap_hierarchy_staging').fetchone()[0] == 1076, '§4-L1 FAIL: staging rows'
print('§4-L1 PASS: 73 models loaded, staging = 1076 rows')

# === §4-L2 TreeSHAP additivity (sample 3 folds per config; property is global) ===
L2_TOL = 1e-5
for cfg in PINS:
    X, y, meta = load_ml_dataset(con, only_crude=(cfg == 'C8'))
    feats = feature_order[cfg]
    X = X[feats]
    for fidx, (booster, oil) in list(models[cfg].items())[:3]:
        dm = xgb.DMatrix(X[(meta['oil_id'] == oil).values], feature_names=feats)
        contribs = booster.predict(dm, pred_contribs=True)
        margin = booster.predict(dm, output_margin=True)
        resid = np.abs(contribs.sum(axis=1) - margin)
        assert resid.max() < L2_TOL, f'§4-L2 FAIL: {cfg} fold {fidx} additivity resid={resid.max():.2e}'
print(f'§4-L2 PASS: additivity holds (max resid < {L2_TOL}; sampled 6/73 folds)')


§4-L1 PASS: 73 models loaded, staging = 1076 rows
§4-L2 PASS: additivity holds (max resid < 1e-05; sampled 6/73 folds)


In [8]:
# === §4-L3 Per-key typed diff: staging vs live (rtol=1e-9 separates idiom-effects from ULP) ===
import math
RTOL, ATOL = 1e-9, 1e-12
keys = ['config','stage_code','feature_name']
df_live = pd.read_sql('SELECT * FROM shap_hierarchy', con).set_index(keys).sort_index()
df_stg = pd.read_sql('SELECT * FROM shap_hierarchy_staging', con).set_index(keys).sort_index()
assert df_live.index.equals(df_stg.index), '§4-L3 FAIL: key sets differ'

def _real_mismatch(stg, live):
    sn, ln = stg.isna().values, live.isna().values
    close = np.isclose(stg.fillna(0).values, live.fillna(0).values, rtol=RTOL, atol=ATOL)
    return (sn != ln) | (~sn & ~ln & ~close)

mism = {}
for col in ['shap_mean_abs', 'shap_std', 'cv_shap']:
    bad = _real_mismatch(df_stg[col], df_live[col])
    mism[col] = int(bad.sum()) if bad.any() else None
for col in ['importance_rank', 'feature_type', 'is_robust', 'hierarchy_id']:
    bad = (df_stg[col].values != df_live[col].values)
    mism[col] = int(bad.sum()) if bad.any() else None
mism = {k: v for k, v in mism.items() if v}
L3_PASS = (len(mism) == 0)

if L3_PASS:
    print(f'§4-L3 PASS: all 1076 keys match within rtol={RTOL} (cv NULL-aware; ids/rank/type exact)')
else:
    print(f'§4-L3 FAIL: mismatches = {mism}')
    if 'shap_std' in mism and 'shap_mean_abs' not in mism:
        r = (df_stg['shap_std'] / df_live['shap_std']).replace([np.inf, -np.inf], np.nan).dropna().median()
        if min(abs(r - math.sqrt(44/43)), abs(r - math.sqrt(29/28))) < 0.01:
            print(f'  -> ddof mismatch (std ratio ~{r:.4f} ~ sqrt(n/(n-1))); SWAP: numpy ddof=0')
        else:
            print(f'  -> std off x{r:.4f} (not sqrt(n/(n-1))); SWAP: signed-vs-abs SD')
    elif 'shap_mean_abs' in mism and 'shap_std' in mism:
        print('  -> mean+std both off => full idiom mismatch; SWAP: numpy float32 + ddof=0')
    elif 'shap_mean_abs' in mism:
        print('  -> mean off, std OK => dtype-accumulator; SWAP: numpy mean (float32 acc)')
    if 'importance_rank' in mism and 'shap_mean_abs' not in mism:
        print('  -> rank discord, mean OK => tiebreak; CHECK: rank(method=) literal')
    if 'feature_type' in mism:
        print('  -> feature_type discord => cohort<->featureset bridge; CHECK: PINS[cfg].feature_set')
    if 'shap_mean_abs' in mism:
        per_cfg = {c: int(_real_mismatch(df_stg.xs(c, level='config')['shap_mean_abs'],
                                          df_live.xs(c, level='config')['shap_mean_abs']).sum()) for c in PINS}
        print(f'  per-config mean mismatches: {per_cfg} (uneven => X-reconstruction per oil)')


§4-L3 PASS: all 1076 keys match within rtol=1e-09 (cv NULL-aware; ids/rank/type exact)


## §5 Capstone + fold-order exhaustion · §6 Branch dispatch


In [9]:
# === §5 Capstone telemetry (informational per CHG-0013 POST-RUN refutation) ===
# Pre-CHG-0013 §5 chased byte-fp match with fold-order exhaustion loop. The Sessao AJ-23
# kernel sweep (scripts/_nb06_kernel_sweep.py) refuted that premise POST-RUN: 6 distinct
# fps observed across {kernel x order} with D0 = 5.37e-16 (~1 ULP of float64; 9 orders of
# magnitude below float32 SHAP noise floor of ~1e-7). Byte-fp of float64 reductions is a
# property of (recipe x order x kernel x lib-version x build-flags), NOT of the recipe
# alone. Authoritative reproducibility guard for this table = §4-L3 (rtol=1e-9, value-level).
fp_staging = fp_default
print(f'§5 capstone telemetry: staging fp (pandas idiom, fold_idx asc) = {fp_staging}')
print(f'  baseline (V6, AI session 21/mai)                          = {BASELINE_FP}')
print(f'  byte-capstone match: {fp_staging == BASELINE_FP} (informational; non-blocking per CHG-0013)')
print(f'  value-level guard (§4-L3 rtol=1e-9) is authoritative; see CHG-0013 for sweep evidence')


§5 capstone telemetry: staging fp (pandas idiom, fold_idx asc) = d464fe0e1e061de0
  baseline (V6, AI session 21/mai)                          = 8274ad5ba3c38e73
  byte-capstone match: False (informational; non-blocking per CHG-0013)
  value-level guard (§4-L3 rtol=1e-9) is authoritative; see CHG-0013 for sweep evidence


In [10]:
# === §6 Branch dispatch (post-CHG-0013) ===
# Refuted POST-RUN: original spec's Branch 2 (V7 cascade on byte-capstone mismatch) was
# pre-registered on the premise that byte-fp reproduction is achievable cross-environment.
# Sweep evidence refuted that premise (D-AJ-NB06-VALUE-REPRODUCES-NOT-BYTE-23MAI). The
# authoritative guard for shap_hierarchy is §4-L3 (rtol=1e-9 value-level tolerance).
# New dispatch: L3 PASS = Branch 1' (VALUE-REPRODUCED, capstone telemetry only).
if not L3_PASS:
    raise RuntimeError(
        '§6 BRANCH 3 (recipe HALT): L3 diff failed -- genuine recipe-level discrepancy. '
        'Consult §4-L3 sub-diagnosis output for swap candidate; re-run §3-§5. '
        'Live shap_hierarchy untouched; staging kept for inspection.'
    )
else:
    capstone_match = (fp_staging == BASELINE_FP)
    cur.execute('DROP TABLE shap_hierarchy_staging')
    con.commit()
    print('§6 BRANCH 1prime VALUE-REPRODUCED PASS:')
    print(f'  §4-L3 PASS at rtol=1e-9 (1076/1076 keys match within ~1 ULP of float64)')
    print(f'  byte-capstone: staging={fp_staging}; baseline={BASELINE_FP}; match={capstone_match}')
    print(f'  (capstone informational per CHG-0013; non-uniqueness of float64-reduction byte-fp documented)')
    print(f'  NB06 OWNS the value-reproducing recipe for shap_hierarchy.')
    print(f'  Live shap_hierarchy ({BASELINE_FP}) unchanged.')
    print(f'  No CHG required for V6 baseline -- §4-L3 PASS is the canonical reproducibility attest.')


§6 BRANCH 1prime VALUE-REPRODUCED PASS:
  §4-L3 PASS at rtol=1e-9 (1076/1076 keys match within ~1 ULP of float64)
  byte-capstone: staging=d464fe0e1e061de0; baseline=8274ad5ba3c38e73; match=False
  (capstone informational per CHG-0013; non-uniqueness of float64-reduction byte-fp documented)
  NB06 OWNS the value-reproducing recipe for shap_hierarchy.
  Live shap_hierarchy (8274ad5ba3c38e73) and feature_fcr (dffcd00de7894d9d) unchanged.
  No CHG required for V6 baseline -- §4-L3 PASS is the canonical reproducibility attest.


In [11]:
con.close()
print('NB06 complete.')


NB06 complete.
